# Skin Cancer Classification using Custom CNN Models (TensorFlow/Keras)

**Dataset:** Skin Cancer (Benign vs Malignant) from GitHub

**Repository:** https://github.com/IamSamk/DL.git

**Optimized for:** Google Colab / Kaggle

## Setup GPU and Mixed Precision

In [ ]:
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
print('GPU Available:', tf.config.list_physical_devices('GPU'))

# Enable mixed precision
from tensorflow.keras import mixed_precision
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print('Mixed precision enabled:', policy.name)

## Clone Dataset (Sparse Checkout - Only skin_dataset_resized folder)

In [ ]:
import os

repo_url = 'https://github.com/IamSamk/DL.git'
dataset_folder = 'skin_dataset_resized'

if not os.path.exists(f'/content/{dataset_folder}'):
    print(f'Cloning only {dataset_folder} folder...')
    !git clone --depth 1 --filter=blob:none --sparse {repo_url} /content/DL_temp
    os.chdir('/content/DL_temp')
    !git sparse-checkout set {dataset_folder}
    !mv {dataset_folder} /content/
    os.chdir('/content')
    !rm -rf DL_temp
    print(f'✓ {dataset_folder} cloned')
else:
    print(f'✓ {dataset_folder} already exists')

## Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

np.random.seed(42)
tf.random.set_seed(42)

print('✓ Libraries imported')

## Configure Paths (Auto-detect Environment)

In [ ]:
# Auto-detect environment
if os.path.exists('/content/skin_dataset_resized'):
    base_dir = '/content/skin_dataset_resized'
    print('Environment: Colab')
elif os.path.exists('/kaggle/working'):
    if not os.path.exists('/kaggle/working/skin_dataset_resized'):
        print('Cloning dataset...')
        !git clone --depth 1 --filter=blob:none --sparse https://github.com/IamSamk/DL.git /kaggle/temp
        os.chdir('/kaggle/temp')
        !git sparse-checkout set skin_dataset_resized
        !mv skin_dataset_resized /kaggle/working/
        os.chdir('/kaggle/working')
        !rm -rf /kaggle/temp
    base_dir = '/kaggle/working/skin_dataset_resized'
    print('Environment: Kaggle')
else:
    base_dir = r'c:\Users\Samarth Kadam\Documents\DL\skin_dataset_resized'
    print('Environment: Local')

train_dir = os.path.join(base_dir, 'train_set')
val_dir = os.path.join(base_dir, 'val_set')
test_dir = os.path.join(base_dir, 'test_set')

print(f'Dataset: {base_dir}')
for path, name in [(train_dir, 'Train'), (val_dir, 'Val'), (test_dir, 'Test')]:
    if os.path.exists(path):
        print(f'✓ {name} found')
    else:
        print(f'✗ {name} NOT found')